# 02 - Data Preparation : IEEE-CIS Fraud Detection

**Étape CRISP-DM : Data Preparation**

Objectif : produire un dataset propre et exploitable pour la modélisation, via le module
`src/pipeline.py`.

Décisions de préparation (justifiées dans `docs/`) :
- Les valeurs manquantes numériques ne sont **pas imputées** : XGBoost/LightGBM gèrent nativement
  les NaN en apprenant la direction de split optimale, et l'EDA a montré que le "manquant" porte
  parfois un signal (ex. `DeviceType`).
- Un flag `has_identity_data` capture explicitement la couverture identity (23,8%) avant qu'elle
  ne se dilue dans 214 colonnes individuellement incomplètes.
- Les colonnes catégorielles sont encodées en codes entiers (label encoding), adapté aux modèles
  à arbres et sans explosion dimensionnelle.
- Le résultat est sauvegardé en Parquet (`data/processed/`) pour un rechargement rapide.

In [1]:
import os
import sys

sys.path.append(os.path.join(".."))

from src.pipeline import load_and_prepare_data, save_processed, save_mappings, get_column_types

DATA_DIR = os.path.join("..", "data", "raw")
TRANSACTION_PATH = os.path.join(DATA_DIR, "train_transaction.csv")
IDENTITY_PATH = os.path.join(DATA_DIR, "train_identity.csv")
OUTPUT_PATH = os.path.join("..", "data", "processed", "train_clean.parquet")
MAPPINGS_PATH = os.path.join("..", "data", "processed", "category_mappings.pkl")

In [2]:
df, mappings = load_and_prepare_data(TRANSACTION_PATH, IDENTITY_PATH, encode=True)
print(f"Shape finale : {df.shape}")

mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memoire occupee : {mem_mb:.1f} Mo")

categorical_cols, numerical_cols = get_column_types(df)
print(f"Colonnes categorielles encodees : {len(categorical_cols)}")
print(f"Colonnes numeriques (NaN natifs conserves) : {len(numerical_cols)}")
print(f"Couverture identity (flag has_identity_data) : {df['has_identity_data'].mean():.1%}")

Shape finale : (590540, 435)
Memoire occupee : 976.6 Mo
Colonnes categorielles encodees : 60
Colonnes numeriques (NaN natifs conserves) : 372
Couverture identity (flag has_identity_data) : 23.8%


In [3]:
save_processed(df, OUTPUT_PATH)
save_mappings(mappings, MAPPINGS_PATH)

csv_size_mb = os.path.getsize(TRANSACTION_PATH) / 1024**2
parquet_size_mb = os.path.getsize(OUTPUT_PATH) / 1024**2
print(f"Taille train_transaction.csv : {csv_size_mb:.1f} Mo")
print(f"Taille {os.path.basename(OUTPUT_PATH)} : {parquet_size_mb:.1f} Mo")
print(f"Ratio de compression : {csv_size_mb / parquet_size_mb:.1f}x")
print(f"Mappings categoriels sauvegardes : {len(mappings)} colonnes ({os.path.basename(MAPPINGS_PATH)})")

Taille train_transaction.csv : 651.7 Mo
Taille train_clean.parquet : 82.1 Mo
Ratio de compression : 7.9x
Mappings categoriels sauvegardes : 60 colonnes (category_mappings.pkl)


In [4]:
import time

import pandas as pd

start = time.time()
df_reload = pd.read_parquet(OUTPUT_PATH)
elapsed = time.time() - start
print(f"Rechargement depuis Parquet : {elapsed:.2f}s pour {df_reload.shape}")
assert df_reload.shape == df.shape, "Le rechargement ne correspond pas aux donnees sauvegardees"

Rechargement depuis Parquet : 1.33s pour (590540, 435)
